# Python. Семинар 5. Функции и классы вблизи


## 1. Присваивание делает имя локальным во всей функции

В лекции показывали `global`: без него функция не может присвоить значение
глобальной переменной. Посмотрим, что происходит, если про `global` забыть.
Сначала — чтение, оно работает и без объявлений:


In [ ]:
total = 10


def show() -> int:
    return total


show()

Теперь то же имя, но с присваиванием:


In [ ]:
total = 10


def increase() -> None:
    total = total + 1  # noqa: F823, F841


increase()

Ошибка не «переменной нет», а «локальная переменная ещё не получила значения».

Локальность имени определяется **при разборе функции**, а не при выполнении:
раз в теле есть `total = ...`, имя локально во всей функции, включая строки
до присваивания. В `total + 1` читается несозданная локальная переменная, а не
глобальная десятка. Линтер находит это без запуска — отсюда `# noqa: F823`.

Лечится объявлением `global` или, что лучше, отказом от глобального состояния:
значение приходит аргументом и возвращается результатом.


In [ ]:
def increased(value: int) -> int:
    return value + 1


total = increased(total)
total

**Изменение** объекта присваиванием не является и `global` не требует: список
меняется на месте, имя ни к чему новому не привязывается.


In [ ]:
items: list[int] = []


def remember(value: int) -> None:
    items.append(value)


remember(1)
remember(2)
items

То же различие между привязкой имени и изменением объекта, что и на прошлом
семинаре, — теперь на границе функции.


## 2. Аннотации типов ничего не проверяют

Легко решить, что аннотации защищают от неверных аргументов. Не защищают: при
выполнении Python их игнорирует.


In [ ]:
def double(x: int) -> int:
    return x * 2


double("ку")

Строка умножилась на два. Аннотация — не проверка, а **запись о намерении**:
для читателя и для `mypy`, который разбирает код не запуская и на
`double("ку")` ругается заранее.

Сами аннотации складываются в обычный словарь — атрибут функции.


In [ ]:
double.__annotations__

Если аргумент нужно проверить, это делает код, а не аннотация. На скорость и
поведение аннотации не влияют, поэтому писать их можно везде.


## 3. Список, объявленный в классе, — один на всех

В лекции у класса было поле `var = 0`, общее для всех экземпляров. С числом это
безобидно. Заменим его на список:


In [ ]:
class Basket:
    items: list[str] = []

    def add(self, item: str) -> None:
        self.items.append(item)


first = Basket()
second = Basket()

first.add("яблоко")
first.items, second.items

Яблоко лежит в обеих корзинах, хотя `add` вызывали только у первой. Список
создан **один раз** — в момент выполнения тела класса, — и принадлежит самому
классу. Обращение `self.items` не находит ничего у объекта и уходит искать
дальше, в класс, поэтому оба объекта работают с одним и тем же списком:


In [ ]:
first.items is second.items, first.__dict__

Словарь объекта пуст: своего `items` у него никогда и не было.

Правило простое: **изменяемое состояние создаётся в `__init__`**, а не в теле
класса. Тогда у каждого объекта список свой.


In [ ]:
class Basket:
    def __init__(self) -> None:
        self.items: list[str] = []

    def add(self, item: str) -> None:
        self.items.append(item)


first = Basket()
second = Basket()

first.add("яблоко")
first.items, second.items, first.items is second.items

Заметьте, что с числом ошибка не видна, но по той же причине: `self.var += 1`
не меняет число на месте, а создаёт объекту **своё** поле, затеняющее поле
класса. Именно этот конфликт имён разбирался в лекции.


## 4. `obj.method` — это объект, который помнит `obj`

В лекции метод вызывали и через объект, и через класс: `a.die()` и
`SampleClass.die(a)`. Посмотрим, что получится, если метод не вызвать, а просто
взять.


In [ ]:
basket = Basket()
add_to_basket = basket.add

add_to_basket("груша")
basket.items

Скобок при взятии не было, `basket` нигде не упоминался — а груша попала в его
корзину. `basket.add` возвращает не объявленную в классе функцию, а **связанный
метод**: обёртку, внутри которой уже записан `self`.


In [ ]:
type(basket.add).__name__, type(Basket.add).__name__

In [ ]:
basket.add.__self__ is basket

Связанный метод — обычное значение: его кладут в список или словарь, передают в
другую функцию и вызывают позже, не таская объект отдельно.


## 5. Словарь функций вместо цепочки `if`

Разбор команды обычно пишут цепочкой условий:

```python
if name == "плюс":
    result = a + b
elif name == "минус":
    result = a - b
elif name == "умножить":
    result = a * b
```

Раз функция — такой же объект, как число или строка, её можно положить в
словарь значением. Тогда выбор ветки превращается в обычный поиск по ключу:


In [ ]:
from collections.abc import Callable


def add(a: int, b: int) -> int:
    return a + b


def subtract(a: int, b: int) -> int:
    return a - b


OPERATIONS: dict[str, Callable[[int, int], int]] = {"плюс": add, "минус": subtract}

OPERATIONS["плюс"](2, 3)

Выигрыш не только в длине. Список поддерживаемых команд теперь можно получить
как данные, а добавление новой операции не требует править разбор:


In [ ]:
sorted(OPERATIONS)

In [ ]:
OPERATIONS["умножить"] = lambda a, b: a * b

sorted(OPERATIONS), OPERATIONS["умножить"](2, 3)

Значением словаря может быть и связанный метод из прошлого раздела — тогда
таблица команд будет работать с состоянием конкретного объекта.
